# Strict-AOG v18 revised diagnostics

This notebook fuses the v14-style diagnostic generator with the revised diagnostic analysis for the `strict_aog` v18 model.

1. **Generate v18 diagnostics** from the v17 grammar/cache used by v18 training, plus the best v18 checkpoint into `runs/strict_aog_v18_raw_count_role_floor_diagnostics`.
2. **Analyze the generated diagnostics** with the revised v14 diagnostic checks, comparing against v17 diagnostics when available.

Run the generation section first to create the diagnostic folder, then run the revised analysis section. The analysis section also works if `runs/strict_aog_v18_raw_count_role_floor_diagnostics` or its `.zip` already exists.


In [ ]:
from pathlib import Path
import sys, json, math, csv, shutil
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from torch.utils.data import DataLoader


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'runs').exists() and (candidate / 'src' / 'partcat_hkg').exists():
            return candidate
    return start


REPO_ROOT = find_repo_root()
RUNS_DIR = REPO_ROOT / 'runs'
GRAMMAR_PATH = RUNS_DIR / 'strict_aog_cache_v17' / 'strict_aog_v17.pt'
VAL_CACHE_PATH = RUNS_DIR / 'strict_aog_cache_v17' / 'val_strict_aog_terminals.pt'
CHECKPOINT_PATH = RUNS_DIR / 'strict_aog_v18_raw_count_role_floor' / 'checkpoints' / 'strict_aog_best.pt'  # set to None if untrained
DIAGNOSTIC_DIR = RUNS_DIR / 'strict_aog_v18_raw_count_role_floor_diagnostics'
OUT_DIR = DIAGNOSTIC_DIR
RESET_DIAGNOSTIC_DIR = False
SHOW_GENERATION_FIGURES = False
MAX_WRONG_OVERLAYS = 12
MAX_EDGE_AUDIT_WRONG = 50
BATCH_SIZE = 128 if torch.cuda.is_available() else 16
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

SRC = REPO_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from partcat_hkg.strict_aog.data import StrictAOGTerminalDataset, collate_strict_aog
from partcat_hkg.strict_aog.grammar import load_strict_aog
from partcat_hkg.strict_aog.parser import ParserConfig, StrictAOGParser

if RESET_DIAGNOSTIC_DIR and OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / 'figures').mkdir(exist_ok=True)
(OUT_DIR / 'figures' / 'wrong_overlays').mkdir(parents=True, exist_ok=True)
(OUT_DIR / 'tables').mkdir(exist_ok=True)
print('device', DEVICE)
print('REPO_ROOT:', REPO_ROOT)
print('GRAMMAR_PATH:', GRAMMAR_PATH, GRAMMAR_PATH.exists())
print('VAL_CACHE_PATH:', VAL_CACHE_PATH, VAL_CACHE_PATH.exists())
print('CHECKPOINT_PATH:', CHECKPOINT_PATH, Path(CHECKPOINT_PATH).exists() if CHECKPOINT_PATH else None)
print('DIAGNOSTIC_DIR:', DIAGNOSTIC_DIR)


def finish_generation_figure():
    if SHOW_GENERATION_FIGURES:
        plt.show()
    else:
        plt.close()


In [ ]:
grammar = load_strict_aog(str(GRAMMAR_PATH))
parser_cfg = ParserConfig(
    assignment='gpu_mf',
    beam_size=8,
    top_terminals_per_slot=4,
    class_chunk=0,
    mf_iters=2,
    mf_tau=0.50,
    mf_column_iters=6,
    mf_edge_chunk_size=128,
    relation_weight=0.10,
    role_overlap_weight=0.40,
    min_role_overlap=0.0,
    min_parse_role_overlap=0.20,
    low_role_penalty=0.75,
    min_parse_inst_edges=2.0,
    low_inst_edge_penalty=0.75,
    min_parse_edge_coverage=0.40,
    low_edge_coverage_penalty=0.75,
    count_weight=0.20,
    count_role_power=0.5,
    count_source='assigned',
    count_model='categorical',
    count_score_mode='raw',
    role_overlap_floor=0.02,
    count_ll_clip_min=-8.0,
    count_ll_clip_max=4.0,
    edge_missing_weight=0.75,
    edge_score_mode='peer_llr',
    edge_background_min_count=8.0,
    class_prior_weight=0.0,
    edge_info_gain_power=0.5,
    score_normalization='',
    node_score_normalization='none',
    edge_score_normalization='sqrt',
    node_app_weight=0.30,
    node_geom_weight=0.35,
    node_presence_weight=0.05,
    slot_prior_weight=0.02,
    missing_weight=0.45,
    spurious_weight=0.05,
    template_tau=0.75,
)
model = StrictAOGParser(grammar, parser_cfg).to(DEVICE).eval()
if CHECKPOINT_PATH and Path(CHECKPOINT_PATH).exists():
    ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
    state = ckpt.get('model', ckpt.get('state_dict', ckpt)) if isinstance(ckpt, dict) else ckpt
    model.load_state_dict(state, strict=True)
    print('loaded checkpoint', CHECKPOINT_PATH)
else:
    print('No checkpoint loaded; using grammar statistics + initial calibration.')

eval_ds = StrictAOGTerminalDataset(VAL_CACHE_PATH, include_visual=False, lru_shards=4)
viz_ds = StrictAOGTerminalDataset(VAL_CACHE_PATH, include_visual=True, lru_shards=4)
ds = eval_ds
loader = DataLoader(eval_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, collate_fn=collate_strict_aog)
print('records', len(eval_ds), 'classes', grammar.num_classes, 'templates', grammar.num_templates, 'slots', grammar.max_slots, 'edges', int(grammar.edges.shape[0]))
print('parser', parser_cfg)


## 1. Evaluate parser and collect intermediate scores

In [ ]:
@torch.no_grad()
def evaluate_collect(model, loader):
    rows = []
    offset = 0
    for batch in loader:
        labels = batch["obj_label"].to(DEVICE)
        bsz = labels.shape[0]
        batch_dev = {k: v.to(DEVICE) if torch.is_tensor(v) else v for k, v in batch.items()}
        out = model(batch_dev, enable_edges=True, return_parse=False)
        logits = out["logits"]
        pred = logits.argmax(-1)
        probs = torch.softmax(logits, dim=-1)
        entropy = -(probs * torch.log(probs.clamp_min(1e-9))).sum(-1)
        top2 = torch.topk(logits, k=min(2, logits.shape[-1]), dim=-1).values
        margin = top2[:, 0] - top2[:, 1] if top2.shape[1] > 1 else torch.zeros_like(top2[:, 0])
        b_index = torch.arange(bsz, device=DEVICE)
        best_t_pred = out["best_template"][b_index, pred]
        best_t_true = out["best_template"][b_index, labels]
        node_pred = out["node_scores"][b_index, pred, best_t_pred]
        edge_pred = out["edge_scores"][b_index, pred, best_t_pred]
        count_pred = out["count_scores"][b_index, pred, best_t_pred]
        templ_pred = out["template_scores"][b_index, pred, best_t_pred]
        role_pred = out["role_overlap_scores"][b_index, pred, best_t_pred]
        node_true = out["node_scores"][b_index, labels, best_t_true]
        edge_true = out["edge_scores"][b_index, labels, best_t_true]
        count_true = out["count_scores"][b_index, labels, best_t_true]
        templ_true = out["template_scores"][b_index, labels, best_t_true]
        role_true = out["role_overlap_scores"][b_index, labels, best_t_true]
        weighted_edge_pred = float(parser_cfg.relation_weight) * edge_pred
        weighted_edge_true = float(parser_cfg.relation_weight) * edge_true
        weighted_count_pred = float(parser_cfg.count_weight) * count_pred
        weighted_count_true = float(parser_cfg.count_weight) * count_true
        denom = node_pred.abs() + weighted_edge_pred.abs() + weighted_count_pred.abs() + 1e-6
        for i in range(bsz):
            rec = eval_ds[offset+i]
            sample_index = rec.get("sample_index", torch.tensor(offset+i))
            if torch.is_tensor(sample_index):
                sample_index = int(sample_index.item())
            rows.append({
                "index": int(offset+i),
                "sample_index": int(sample_index),
                "true": int(labels[i].cpu()),
                "true_name": grammar.class_names[int(labels[i])],
                "pred": int(pred[i].cpu()),
                "pred_name": grammar.class_names[int(pred[i])],
                "correct": bool(pred[i].item() == labels[i].item()),
                "pred_template": int(best_t_pred[i].cpu()),
                "true_template": int(best_t_true[i].cpu()),
                "logit_true": float(logits[i, labels[i]].cpu()),
                "logit_pred": float(logits[i, pred[i]].cpu()),
                "margin": float(margin[i].cpu()),
                "entropy": float(entropy[i].cpu()),
                "node_pred": float(node_pred[i].cpu()),
                "edge_pred": float(edge_pred[i].cpu()),
                "count_pred": float(count_pred[i].cpu()),
                "role_overlap_pred": float(role_pred[i].cpu()),
                "weighted_edge_pred": float(weighted_edge_pred[i].cpu()),
                "weighted_count_pred": float(weighted_count_pred[i].cpu()),
                "template_pred": float(templ_pred[i].cpu()),
                "node_true": float(node_true[i].cpu()),
                "edge_true": float(edge_true[i].cpu()),
                "count_true": float(count_true[i].cpu()),
                "role_overlap_true": float(role_true[i].cpu()),
                "weighted_edge_true": float(weighted_edge_true[i].cpu()),
                "weighted_count_true": float(weighted_count_true[i].cpu()),
                "template_true": float(templ_true[i].cpu()),
                "edge_fraction_abs": float(weighted_edge_pred[i].abs().cpu() / denom[i].cpu()),
                "count_fraction_abs": float(weighted_count_pred[i].abs().cpu() / denom[i].cpu()),
                "edge_missing_mean": float(out["edge_missing_count"].mean().cpu()),
                "assignment_reuse_mean": float(out["assignment_reuse_mean"].cpu()),
                "instantiated_edge_mean": float(out["instantiated_edge_mean"].cpu()),
                "count_score_mean": float(out["count_score_mean"].cpu()),
                "role_overlap_mean": float(out["role_overlap_mean"].cpu()),
            })
        offset += bsz
    return pd.DataFrame(rows)

pred_df = evaluate_collect(model, loader)
pred_df.to_csv(OUT_DIR / "tables" / "predictions.csv", index=False)
acc = pred_df["correct"].mean()
print("accuracy", acc, "n", len(pred_df), "wrong", int((~pred_df.correct).sum()))
pred_df.head()


## 2. Summary plots

In [ ]:
def savefig(name):
    path = OUT_DIR / "figures" / name
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    print("saved", path)

# Confusion matrix
C = grammar.num_classes
cm = np.zeros((C, C), dtype=int)
for _, r in pred_df.iterrows():
    cm[int(r.true), int(r.pred)] += 1
cm_norm = cm / np.maximum(cm.sum(1, keepdims=True), 1)
plt.figure(figsize=(8, 7))
plt.imshow(cm_norm, aspect="auto")
plt.colorbar(label="row-normalized count")
plt.xticks(range(C), grammar.class_names, rotation=90)
plt.yticks(range(C), grammar.class_names)
plt.xlabel("predicted")
plt.ylabel("true")
plt.title("Strict AOG v18 confusion matrix")
savefig("confusion_matrix.png")
finish_generation_figure()

# Node vs weighted edge scatter
plt.figure(figsize=(6, 5))
mask = pred_df.correct.values.astype(bool)
plt.scatter(pred_df.loc[mask, "node_pred"], pred_df.loc[mask, "weighted_edge_pred"], s=12, label="correct", alpha=0.65)
plt.scatter(pred_df.loc[~mask, "node_pred"], pred_df.loc[~mask, "weighted_edge_pred"], s=24, label="wrong", alpha=0.9)
plt.axhline(0, linewidth=0.8)
plt.axvline(0, linewidth=0.8)
plt.xlabel("node score of predicted parse")
plt.ylabel("weighted edge contribution")
plt.title("Node vs edge contribution")
plt.legend()
savefig("node_vs_edge_scatter.png")
finish_generation_figure()

# v17 count/cardinality contribution
plt.figure(figsize=(6, 5))
plt.scatter(pred_df.loc[mask, "weighted_edge_pred"], pred_df.loc[mask, "weighted_count_pred"], s=12, label="correct", alpha=0.65)
plt.scatter(pred_df.loc[~mask, "weighted_edge_pred"], pred_df.loc[~mask, "weighted_count_pred"], s=24, label="wrong", alpha=0.9)
plt.axhline(0, linewidth=0.8)
plt.axvline(0, linewidth=0.8)
plt.xlabel("weighted edge contribution")
plt.ylabel("weighted count contribution")
plt.title("Edge vs count contribution")
plt.legend()
savefig("edge_vs_count_scatter.png")
finish_generation_figure()

plt.figure(figsize=(7, 4))
plt.hist(pred_df.loc[mask, "weighted_count_pred"], bins=30, alpha=0.65, label="correct")
plt.hist(pred_df.loc[~mask, "weighted_count_pred"], bins=30, alpha=0.75, label="wrong")
plt.axvline(0, linewidth=0.8)
plt.xlabel("weighted count contribution")
plt.ylabel("images")
plt.title("Part-count evidence distribution")
plt.legend()
savefig("count_contribution_hist.png")
finish_generation_figure()

plt.figure(figsize=(7, 4))
plt.hist(pred_df.loc[mask, "count_fraction_abs"], bins=30, alpha=0.65, label="correct")
plt.hist(pred_df.loc[~mask, "count_fraction_abs"], bins=30, alpha=0.75, label="wrong")
plt.xlabel("absolute count share of |node|+|edge|+|count|")
plt.ylabel("images")
plt.title("Relative count contribution")
plt.legend()
savefig("count_fraction_hist.png")
finish_generation_figure()

# Template usage heatmap
usage = np.zeros((C, grammar.num_templates), dtype=int)
for _, r in pred_df.iterrows():
    usage[int(r.pred), int(r.pred_template)] += 1
plt.figure(figsize=(8, 4))
plt.imshow(usage, aspect="auto")
plt.colorbar(label="count")
plt.xticks(range(grammar.num_templates), [f"T{a}" for a in range(grammar.num_templates)])
plt.yticks(range(C), grammar.class_names)
plt.title("Predicted class/template usage")
savefig("template_usage_heatmap.png")
finish_generation_figure()

# Prediction distribution
vc = pred_df.pred_name.value_counts()
plt.figure(figsize=(8, 4))
plt.bar(range(len(vc)), vc.values)
plt.xticks(range(len(vc)), vc.index, rotation=90)
plt.ylabel("count")
plt.title("Prediction distribution")
savefig("prediction_distribution.png")
finish_generation_figure()


## 2b. Category-template AOG structure

Visualize the learned Or-node category and its template And-nodes: slot children are labeled by part, and horizontal AOG relation factors are drawn between slots.


In [ ]:
# Pick a category name or integer class index. Set TEMPLATE_ID=None to show every valid template.
CATEGORY_TEMPLATE_CLASS = "car"
TEMPLATE_ID = None

EDGE_TYPE_LABEL = {0: "anchor", 1: "repeat", 2: "generic"}
EDGE_TYPE_COLOR = {0: "#2563eb", 1: "#7c3aed", 2: "#475569"}


def _safe_stem(text):
    return "".join(ch if ch.isalnum() or ch in "._-" else "_" for ch in str(text)).strip("_") or "category"


def category_to_index(category):
    if isinstance(category, (int, np.integer)):
        c = int(category)
    else:
        names = [str(x) for x in grammar.class_names]
        if str(category) not in names:
            raise ValueError(f"Unknown category {category!r}; available: {names}")
        c = names.index(str(category))
    if c < 0 or c >= int(grammar.num_classes):
        raise ValueError(f"Category index out of range: {c}")
    return c


def template_edge_indices(c, a):
    edges = grammar.edges.detach().cpu().long().numpy()
    if edges.size == 0:
        return []
    return np.where((edges[:, 0] == int(c)) & (edges[:, 1] == int(a)))[0].astype(int).tolist()


def template_structure_rows(c):
    rows = []
    for a in range(int(grammar.num_templates)):
        if float(grammar.template_valid[c, a].detach().cpu()) <= 0:
            continue
        slot_valid = grammar.slot_valid[c, a].detach().cpu().numpy() > 0.5
        slot_parts = grammar.slot_part[c, a].detach().cpu().numpy().astype(int)
        edge_idxs = template_edge_indices(c, a)
        part_counts = {}
        for s, ok in enumerate(slot_valid):
            if not ok:
                continue
            p = int(slot_parts[s])
            pname = grammar.part_names[p] if 0 <= p < len(grammar.part_names) else str(p)
            part_counts[pname] = part_counts.get(pname, 0) + 1
        rows.append({
            "class": grammar.class_names[c],
            "class_idx": int(c),
            "template": int(a),
            "prior": float(grammar.template_prior[c, a].detach().cpu()),
            "num_slots": int(slot_valid.sum()),
            "num_required": int((grammar.slot_required[c, a].detach().cpu().numpy()[slot_valid] > 0.5).sum()),
            "num_edges": len(edge_idxs),
            "parts": ", ".join(f"{k}:{v}" for k, v in sorted(part_counts.items())),
        })
    return rows


def _slot_positions(c, a, valid_slots):
    geom = grammar.slot_geom_mean[c, a].detach().cpu().float().numpy()
    xy = np.stack([geom[:, 0], geom[:, 1]], axis=1)
    finite = np.isfinite(xy).all(axis=1)
    usable = valid_slots & finite
    # If the grammar stores degenerate centers, fall back to a compact circle.
    if usable.sum() == 0 or np.nanmax(np.abs(xy[usable])) < 1e-6:
        idxs = np.where(valid_slots)[0]
        theta = np.linspace(0, 2 * np.pi, max(len(idxs), 1), endpoint=False)
        xy = np.zeros_like(xy)
        for k, s in enumerate(idxs):
            xy[s, 0] = 0.5 + 0.38 * np.cos(theta[k])
            xy[s, 1] = 0.5 + 0.32 * np.sin(theta[k])
    else:
        xy[:, 0] = np.clip(xy[:, 0], 0.04, 0.96)
        xy[:, 1] = np.clip(xy[:, 1], 0.04, 0.96)
    return xy


def draw_template_axis(ax, c, a):
    valid_slots = grammar.slot_valid[c, a].detach().cpu().numpy() > 0.5
    slot_parts = grammar.slot_part[c, a].detach().cpu().numpy().astype(int)
    required = grammar.slot_required[c, a].detach().cpu().numpy() > 0.5
    presence = grammar.slot_presence[c, a].detach().cpu().numpy()
    xy = _slot_positions(c, a, valid_slots)
    edge_idxs = template_edge_indices(c, a)
    edge_rows = grammar.edges.detach().cpu().long().numpy()
    edge_info = grammar.edge_info_gain.detach().cpu().numpy() if getattr(grammar, "edge_info_gain", None) is not None else np.zeros(len(edge_rows))
    edge_support = grammar.edge_support.detach().cpu().numpy() if getattr(grammar, "edge_support", None) is not None else np.zeros(len(edge_rows))
    edge_type = grammar.edge_type.detach().cpu().numpy() if getattr(grammar, "edge_type", None) is not None else np.zeros(len(edge_rows), dtype=int)

    for e in edge_idxs:
        _, _, si, sj = [int(x) for x in edge_rows[e]]
        if si >= len(valid_slots) or sj >= len(valid_slots) or not (valid_slots[si] and valid_slots[sj]):
            continue
        et = int(edge_type[e]) if e < len(edge_type) else 2
        color = EDGE_TYPE_COLOR.get(et, "#475569")
        ax.plot([xy[si, 0], xy[sj, 0]], [xy[si, 1], xy[sj, 1]], color=color, linewidth=1.8, alpha=0.78, zorder=1)
        mx, my = (xy[si] + xy[sj]) / 2.0
        ax.text(mx, my, f"{EDGE_TYPE_LABEL.get(et, et)}\nsup={edge_support[e]:.2f}\nig={edge_info[e]:.2f}",
                ha="center", va="center", fontsize=7,
                bbox=dict(facecolor="white", alpha=0.72, edgecolor="none", pad=1.2), zorder=3)

    for s in np.where(valid_slots)[0]:
        p = int(slot_parts[s])
        pname = grammar.part_names[p] if 0 <= p < len(grammar.part_names) else str(p)
        size = 420 + 820 * float(np.clip(presence[s], 0.0, 1.0))
        ax.scatter([xy[s, 0]], [xy[s, 1]], s=size, marker="o", c="#f8fafc",
                   edgecolors="#111827" if required[s] else "#94a3b8",
                   linewidths=2.2 if required[s] else 1.2, zorder=4)
        ax.text(xy[s, 0], xy[s, 1], f"s{s}\n{pname}\np={presence[s]:.2f}",
                ha="center", va="center", fontsize=8, zorder=5)

    prior = float(grammar.template_prior[c, a].detach().cpu())
    ax.set_title(f"T{a} prior={prior:.3f} slots={int(valid_slots.sum())} edges={len(edge_idxs)}", fontsize=10)
    ax.set_xlim(0, 1)
    ax.set_ylim(1, 0)
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)


def plot_category_template_structure(category=CATEGORY_TEMPLATE_CLASS, template_id=TEMPLATE_ID, save=True):
    c = category_to_index(category)
    valid_templates = [a for a in range(int(grammar.num_templates)) if float(grammar.template_valid[c, a].detach().cpu()) > 0]
    if template_id is not None:
        valid_templates = [int(template_id)]
    if not valid_templates:
        raise ValueError(f"No valid templates for category {grammar.class_names[c]!r}")

    n = len(valid_templates)
    fig, axes = plt.subplots(1, n, figsize=(max(5.0, 4.8 * n), 4.8), squeeze=False)
    for ax, a in zip(axes.ravel(), valid_templates):
        draw_template_axis(ax, c, a)
    fig.suptitle(f"Strict AOG category-template structure: {grammar.class_names[c]}", y=1.03, fontsize=13)
    handles = [
        plt.Line2D([0], [0], color=EDGE_TYPE_COLOR[k], lw=2, label=v)
        for k, v in EDGE_TYPE_LABEL.items()
    ]
    fig.legend(handles=handles, loc="lower center", ncol=len(handles), frameon=False, bbox_to_anchor=(0.5, -0.02))
    fig.tight_layout(rect=[0, 0.06, 1, 0.96])
    if save:
        suffix = "all_templates" if template_id is None else f"template_{int(template_id)}"
        path = OUT_DIR / "figures" / f"aog_category_template_structure_{_safe_stem(grammar.class_names[c])}_{suffix}.png"
        fig.savefig(path, dpi=180, bbox_inches="tight")
        print("saved", path)
    return fig


template_structure_df = pd.DataFrame(template_structure_rows(category_to_index(CATEGORY_TEMPLATE_CLASS)))
template_structure_df.to_csv(OUT_DIR / "tables" / f"aog_category_template_structure_{_safe_stem(CATEGORY_TEMPLATE_CLASS)}.csv", index=False)
display(template_structure_df)
fig = plot_category_template_structure()
finish_generation_figure()


## 3. Decode parse graphs for wrong samples

In [ ]:
@torch.no_grad()
def decode_one(index: int):
    rec = eval_ds[index]
    batch = collate_strict_aog([rec])
    batch_dev = {k: v.to(DEVICE) if torch.is_tensor(v) else v for k, v in batch.items()}
    out = model(batch_dev, enable_edges=True, return_parse=True)
    pred = int(out["logits"].argmax(-1).item())
    true = int(batch["obj_label"].item())
    pg = out["parse_graph"][0]
    return rec, out, pg, true, pred

wrong_df = pred_df[~pred_df.correct].copy().sort_values(["margin"], ascending=False)
wrong_df.to_csv(OUT_DIR / "tables" / "wrong_samples.csv", index=False)
print("num wrong", len(wrong_df))
wrong_df.head(10)

## 4. Overlay assigned terminal masks and AOG relation edges on wrong-classified images

In [ ]:
def tensor_image_to_numpy(img):
    if img is None:
        return None
    x = img.detach().cpu().float()
    if x.ndim == 4:
        x = x[0]
    if x.ndim != 3:
        return None
    if x.shape[0] in (1, 3):
        x = x.permute(1, 2, 0)
    arr = x.numpy()
    # Undo common ImageNet normalization if needed approximately.
    if arr.min() < -0.1 or arr.max() > 1.1:
        arr = arr - arr.min()
        arr = arr / max(arr.max(), 1e-6)
    return np.clip(arr, 0, 1)


def geom_xy(geom, W, H):
    return float(geom[0]) * (W - 1), float(geom[1]) * (H - 1)


def overlay_parse(index: int, save=True):
    rec, out, pg, true, pred = decode_one(index)
    record_raw = viz_ds[index]
    image_tensor = record_raw.get("image")
    if image_tensor is None:
        image_tensor = record_raw.get("image_raw")
    image = tensor_image_to_numpy(image_tensor)
    masks = record_raw.get("terminal_mask")
    geoms = rec["terminal_geom"].detach().cpu().numpy()
    valid = rec["terminal_valid"].detach().cpu().numpy().astype(bool)
    parts = rec["terminal_part"].detach().cpu().numpy()
    if image is None:
        H = W = int(masks.shape[-1]) if masks is not None else 64
        image = np.ones((H, W, 3), dtype=float) * 0.97
    H, W = image.shape[:2]
    fig, ax = plt.subplots(figsize=(7.5, 5.0))
    ax.imshow(image)
    ax.set_title(f"idx={index} true={grammar.class_names[true]} pred={grammar.class_names[pred]} template={pg['template']}\nnode={pg.get('node_score', 0):.2f} edge={pg.get('edge_score', 0):.2f}")
    assigned_terms = set()
    for slot in pg["slots"]:
        if "terminal" not in slot:
            continue
        n = int(slot["terminal"])
        assigned_terms.add(n)
        if masks is not None and n < masks.shape[0]:
            m = masks[n].detach().cpu().float().numpy()
            if m.shape != (H, W):
                # nearest resize without extra deps
                yy = (np.linspace(0, m.shape[0]-1, H)).astype(int)
                xx = (np.linspace(0, m.shape[1]-1, W)).astype(int)
                m = m[yy][:, xx]
            ax.imshow(np.ma.masked_where(m < 0.3, m), alpha=0.35)
        x, y = geom_xy(geoms[n], W, H)
        ax.scatter([x], [y], s=60, marker="o")
        ax.text(x+2, y+2, f"s{slot['slot']}:{slot['part']}\nt{n}", fontsize=8, bbox=dict(facecolor="white", alpha=0.6, edgecolor="none"))
    # draw edges between assigned terminals
    slot_to_term = {int(s["slot"]): int(s["terminal"]) for s in pg["slots"] if "terminal" in s}
    edge_rows = []
    for e in pg["edges"]:
        si, sj = int(e["slot_i"]), int(e["slot_j"])
        ti, tj = slot_to_term.get(si, -1), slot_to_term.get(sj, -1)
        if ti >= 0 and tj >= 0 and ti != tj:
            xi, yi = geom_xy(geoms[ti], W, H)
            xj, yj = geom_xy(geoms[tj], W, H)
            ax.plot([xi, xj], [yi, yj], linewidth=2.0)
            mx, my = (xi+xj)/2, (yi+yj)/2
            lab = f"e{e['edge_index']} ll={e['relation_ll']:.2f}" if e.get("relation_ll") is not None else f"e{e['edge_index']}"
            ax.text(mx, my, lab, fontsize=7, bbox=dict(facecolor="white", alpha=0.55, edgecolor="none"))
            edge_rows.append({"index": index, **e})
    ax.axis("off")
    if save:
        path = OUT_DIR / "figures" / "wrong_overlays" / f"wrong_{index:05d}_{grammar.class_names[true]}_to_{grammar.class_names[pred]}.png"
        fig.savefig(path, dpi=180, bbox_inches="tight")
        print("saved", path)
    return fig, ax

for idx in wrong_df.index[:MAX_WRONG_OVERLAYS].tolist():
    overlay_parse(int(pred_df.loc[idx, "index"]))
    finish_generation_figure()

## 5. Parse validity and relation residual table for wrong samples

In [ ]:
rows = []
for idx in wrong_df["index"].head(MAX_EDGE_AUDIT_WRONG).tolist():
    rec, out, pg, true, pred = decode_one(int(idx))
    slot_to_term = {int(s["slot"]): int(s["terminal"]) for s in pg["slots"] if "terminal" in s}
    for e in pg["edges"]:
        ti = slot_to_term.get(int(e["slot_i"]), -1)
        tj = slot_to_term.get(int(e["slot_j"]), -1)
        rows.append({
            "index": int(idx),
            "true": grammar.class_names[true],
            "pred": grammar.class_names[pred],
            "template": pg["template"],
            "edge_index": e["edge_index"],
            "slot_i": e["slot_i"],
            "slot_j": e["slot_j"],
            "terminal_i": ti,
            "terminal_j": tj,
            "status": e["status"],
            "relation_ll": e["relation_ll"],
            "support": e["support"],
            "self_edge_invalid": bool(ti >= 0 and ti == tj),
        })
edge_df = pd.DataFrame(rows)
edge_df.to_csv(OUT_DIR / "tables" / "wrong_parse_edges.csv", index=False)
if len(edge_df):
    print(edge_df.status.value_counts())
    print("self-edge invalid count", int(edge_df.self_edge_invalid.sum()))
edge_df.head(20)

## 6. Save manifest

In [ ]:
manifest = []
for p in sorted((OUT_DIR / "figures").rglob("*.png")):
    manifest.append({"type": "figure", "path": str(p)})
for p in sorted((OUT_DIR / "tables").rglob("*.csv")):
    manifest.append({"type": "table", "path": str(p)})
manifest_df = pd.DataFrame(manifest)
manifest_df.to_csv(OUT_DIR / "diagnostics_manifest.csv", index=False)
print("saved manifest", OUT_DIR / "diagnostics_manifest.csv")
manifest_df.head()

## Revised analysis of generated diagnostics

The cells below read `runs/strict_aog_v18_raw_count_role_floor_diagnostics` (or its `.zip`) and write additional analysis outputs to `runs/strict_aog_v18_revised_diagnostics/analysis_outputs`.


In [ ]:
from pathlib import Path
import os, zipfile, json, shutil, glob, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'runs').exists() and (candidate / 'src' / 'partcat_hkg').exists():
            return candidate
    return start


REPO_ROOT = find_repo_root()
RUNS_DIR = REPO_ROOT / 'runs'

# Prefer repo-local diagnostics. Each source may be either an extracted directory or a zip.
DIAG_SOURCE = RUNS_DIR / 'strict_aog_v18_raw_count_role_floor_diagnostics'
if not DIAG_SOURCE.exists():
    DIAG_SOURCE = RUNS_DIR / 'strict_aog_v18_raw_count_role_floor_diagnostics.zip'

PREVIOUS_DIAG_SOURCE = RUNS_DIR / 'strict_aog_v17_sweep_relation_w010_count_w020_diagnostics'
if not PREVIOUS_DIAG_SOURCE.exists():
    zip_candidate = RUNS_DIR / 'strict_aog_v17_sweep_relation_w010_count_w020_diagnostics.zip'
    PREVIOUS_DIAG_SOURCE = zip_candidate if zip_candidate.exists() else None

OUT_DIR = RUNS_DIR / 'strict_aog_v18_revised_diagnostics' / 'analysis_outputs'
EXTRACT_DIR = OUT_DIR / 'extracted_v18'
OUT_DIR.mkdir(parents=True, exist_ok=True)
SHOW_ANALYSIS_FIGURES = False


def finish_analysis_figure():
    if SHOW_ANALYSIS_FIGURES:
        plt.show()
    else:
        plt.close()

print('REPO_ROOT:', REPO_ROOT)
print('DIAG_SOURCE:', DIAG_SOURCE, DIAG_SOURCE.exists())
print('PREVIOUS_DIAG_SOURCE:', PREVIOUS_DIAG_SOURCE)
print('OUT_DIR:', OUT_DIR)


In [ ]:
def resolve_diagnostics_base(source, extract_dir):
    source = Path(source)
    candidates = []
    if source.is_dir():
        candidates = [source, *[p for p in source.iterdir() if p.is_dir()]]
        search_root = source
    else:
        if not source.exists():
            raise FileNotFoundError(source)
        if extract_dir.exists():
            shutil.rmtree(extract_dir)
        extract_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(source, 'r') as z:
            z.extractall(extract_dir)
        candidates = [extract_dir, *[p for p in extract_dir.iterdir() if p.is_dir()]]
        search_root = extract_dir

    for base in candidates:
        if (base / 'tables' / 'predictions.csv').exists():
            return base
    files = list(search_root.rglob('predictions.csv'))
    if files:
        return files[0].parents[1]
    raise FileNotFoundError(f'Could not find tables/predictions.csv under {search_root}')


BASE = resolve_diagnostics_base(DIAG_SOURCE, EXTRACT_DIR)
print('BASE:', BASE)

pred = pd.read_csv(BASE / 'tables' / 'predictions.csv')
wrong = pd.read_csv(BASE / 'tables' / 'wrong_samples.csv')
edges_path = BASE / 'tables' / 'wrong_parse_edges.csv'
edges = pd.read_csv(edges_path) if edges_path.exists() else pd.DataFrame()
print('predictions:', pred.shape)
print('wrong:', wrong.shape)
print('wrong_parse_edges:', edges.shape)
display(pred.head())


In [ ]:

# Basic summary.
summary = {
    'n': int(len(pred)),
    'accuracy': float(pred['correct'].mean()),
    'num_wrong': int((~pred['correct']).sum()),
    'num_classes_true': int(pred['true_name'].nunique()),
    'num_classes_pred': int(pred['pred_name'].nunique()),
    'mean_margin': float(pred['margin'].mean()),
    'mean_entropy': float(pred['entropy'].mean()),
    'mean_edge_fraction_abs': float(pred['edge_fraction_abs'].mean()),
    'mean_count_fraction_abs': float(pred['count_fraction_abs'].mean()),
}
print(json.dumps(summary, indent=2))
(OUT_DIR/'summary.json').write_text(json.dumps(summary, indent=2))

class_acc = pred.groupby('true_name').agg(
    n=('correct','size'),
    acc=('correct','mean'),
    pred_most=('pred_name', lambda s: s.value_counts().idxmax()),
).sort_values('acc')
class_acc.to_csv(OUT_DIR/'class_accuracy.csv')
display(class_acc)

confusions = wrong.groupby(['true_name','pred_name']).size().reset_index(name='n').sort_values('n', ascending=False)
confusions.to_csv(OUT_DIR/'top_confusions.csv', index=False)
display(confusions.head(30))


In [ ]:
# Compare against previous diagnostics if available.
def load_pred_from_diagnostics(source):
    if source is None:
        return None
    base = resolve_diagnostics_base(Path(source), OUT_DIR / ('tmp_' + Path(source).stem))
    path = base / 'tables' / 'predictions.csv'
    if not path.exists():
        return None
    return pd.read_csv(path)

if PREVIOUS_DIAG_SOURCE is not None:
    prev = load_pred_from_diagnostics(PREVIOUS_DIAG_SOURCE)
    if prev is None:
        print('No previous predictions.csv found; skipping comparison.')
    else:
        print('previous acc:', prev['correct'].mean(), 'current acc:', pred['correct'].mean())
        m = prev[['index','true_name','pred_name','correct']].merge(
            pred[['index','pred_name','correct']], on='index', suffixes=('_prev','_cur')
        )
        comparison = {
            'previous_acc': float(prev['correct'].mean()),
            'current_acc': float(pred['correct'].mean()),
            'prev_wrong_cur_correct': int(((~m.correct_prev) & m.correct_cur).sum()),
            'prev_correct_cur_wrong': int((m.correct_prev & (~m.correct_cur)).sum()),
            'both_wrong': int(((~m.correct_prev) & (~m.correct_cur)).sum()),
        }
        print(json.dumps(comparison, indent=2))
        new_errors = m[m.correct_prev & (~m.correct_cur)]
        fixed = m[(~m.correct_prev) & m.correct_cur]
        print('New errors introduced by current run:')
        display(new_errors.groupby(['true_name','pred_name_cur']).size().reset_index(name='n').sort_values('n', ascending=False).head(20))
        print('Errors fixed by current run:')
        display(fixed.groupby(['true_name','pred_name_prev']).size().reset_index(name='n').sort_values('n', ascending=False).head(20))
        new_errors.to_csv(OUT_DIR/'new_errors_vs_previous.csv', index=False)
        fixed.to_csv(OUT_DIR/'fixed_errors_vs_previous.csv', index=False)
        (OUT_DIR/'comparison_with_previous.json').write_text(json.dumps(comparison, indent=2))
else:
    print('No previous diagnostic source found; skipping comparison.')



## Score-gap analysis

For each wrong sample, compare the predicted parse score components to the best true-class parse components:

\[
\Delta_{node}=S_{node}^{pred}-S_{node}^{true},\quad
\Delta_{edge}=S_{edge}^{pred}-S_{edge}^{true},\quad
\Delta_{count}=S_{count}^{pred}-S_{count}^{true}.
\]

Positive values mean the wrong class is winning through that component.


In [ ]:

# Add gap columns.
for df in [pred, wrong]:
    for comp in ['node','edge','count','weighted_edge','weighted_count','template','role_overlap']:
        a, b = f'{comp}_pred', f'{comp}_true'
        if a in df.columns and b in df.columns:
            df[f'{comp}_gap'] = df[a] - df[b]

rows=[]
for label, df in [('all', pred), ('correct', pred[pred.correct]), ('wrong', wrong)]:
    row={'subset': label, 'n': len(df)}
    for comp in ['node','edge','count','weighted_edge','weighted_count','template','role_overlap']:
        g=f'{comp}_gap'
        if g in df:
            row[f'{g}_mean']=float(df[g].mean())
            row[f'{g}_median']=float(df[g].median())
            row[f'{g}_positive_frac']=float((df[g]>0).mean())
    row['edge_fraction_abs_mean']=float(df['edge_fraction_abs'].mean()) if len(df) else float('nan')
    row['count_fraction_abs_mean']=float(df['count_fraction_abs'].mean()) if len(df) else float('nan')
    rows.append(row)
gap_summary=pd.DataFrame(rows)
gap_summary.to_csv(OUT_DIR/'gap_summary.csv', index=False)
display(gap_summary.T)

# Plot wrong-sample gaps.
fig, ax = plt.subplots(figsize=(8,5))
cols = [c for c in ['node_gap','weighted_edge_gap','weighted_count_gap','template_gap','role_overlap_gap'] if c in wrong]
wrong[cols].plot(kind='hist', bins=30, alpha=0.55, ax=ax)
ax.axvline(0, linewidth=1)
ax.set_title('Wrong-sample predicted minus true score-component gaps')
ax.set_xlabel('gap')
fig.tight_layout()
fig.savefig(OUT_DIR/'wrong_score_gap_hist.png', dpi=160)
finish_analysis_figure()



## Automatic failure labels

The following heuristic labels are meant to make remaining failure modes more visible:

- **boat_fallback**: predicted class is boat, margin is small, and role overlap is near zero or lower than true. This often means a low-complexity boat template is serving as a fallback parse.
- **edge_driven_wrong**: weighted edge gap is positive enough to change the decision.
- **count_driven_wrong**: count term also favors the wrong class.
- **role_contradiction**: true class has better role overlap but still loses.
- **high_margin_wrong**: model is confidently wrong.


In [ ]:

# Heuristic issue labels.
w = wrong.copy()
w['boat_fallback'] = (w['pred_name'].eq('boat')) & (w['margin'] < 0.35) & ((w['role_overlap_pred'] < 0.05) | (w['role_overlap_gap'] < -0.10))
w['edge_driven_wrong'] = w.get('weighted_edge_gap', 0) > 0.20
w['count_driven_wrong'] = w.get('weighted_count_gap', 0) > 0.03
w['role_contradiction'] = w.get('role_overlap_gap', 0) < -0.10
w['high_margin_wrong'] = w['margin'] > 1.0
w['low_margin_wrong'] = w['margin'] < 0.2
issue_counts = {col:int(w[col].sum()) for col in ['boat_fallback','edge_driven_wrong','count_driven_wrong','role_contradiction','high_margin_wrong','low_margin_wrong']}
print(json.dumps(issue_counts, indent=2))
issue_df = pd.DataFrame([issue_counts])
issue_df.to_csv(OUT_DIR/'issue_counts.csv', index=False)

issue_rows = w[['index','true_name','pred_name','margin','node_gap','weighted_edge_gap','weighted_count_gap','role_overlap_gap','boat_fallback','edge_driven_wrong','count_driven_wrong','role_contradiction','high_margin_wrong','low_margin_wrong','pred_template','true_template']]
issue_rows.to_csv(OUT_DIR/'wrong_failure_labels.csv', index=False)
display(issue_rows.head(30))

# Predicted wrong classes and issue distribution.
issue_by_pred = w.groupby('pred_name').agg(
    n=('correct','size'),
    boat_fallback=('boat_fallback','sum'),
    edge_driven=('edge_driven_wrong','sum'),
    role_contradiction=('role_contradiction','sum'),
    mean_margin=('margin','mean'),
    mean_role_gap=('role_overlap_gap','mean'),
    mean_edge_gap=('weighted_edge_gap','mean'),
).sort_values('n', ascending=False)
issue_by_pred.to_csv(OUT_DIR/'issue_by_pred_class.csv')
display(issue_by_pred)



## Edge table audit

The `wrong_parse_edges.csv` table reports edge statuses for decoded wrong overlays. It is useful for detecting low-complexity templates and missing-edge shortcuts.


In [ ]:

if len(edges):
    print(edges['status'].value_counts(dropna=False))
    edge_by_pair = edges.groupby(['true','pred']).agg(
        rows=('status','size'),
        instantiated=('status', lambda s: int((s=='instantiated').sum())),
        missing=('status', lambda s: int((s!='instantiated').sum())),
        mean_ll=('relation_ll','mean'),
        mean_support=('support','mean')
    ).sort_values('rows', ascending=False)
    edge_by_pair.to_csv(OUT_DIR/'wrong_edge_by_confusion_pair.csv')
    display(edge_by_pair.head(30))

    # Boat-specific wrong edges reveal whether boat is acting as a low-edge fallback.
    boat_edges = edges[edges['pred'].eq('boat')]
    if len(boat_edges):
        print('Boat-predicted wrong edge rows:', len(boat_edges))
        display(boat_edges.head(30))
        boat_edges.to_csv(OUT_DIR/'boat_wrong_edges.csv', index=False)
else:
    print('No wrong_parse_edges.csv available.')


In [ ]:

# Figures: confusion and distributions from saved CSV.
classes = list(pd.Index(sorted(pred['true_name'].unique())))
# Keep dataset order based on true class first occurrence for readability.
classes = list(dict.fromkeys(pred['true_name'].tolist()))
idx = {c:i for i,c in enumerate(classes)}
cm = np.zeros((len(classes), len(classes)), dtype=int)
for _,r in pred.iterrows():
    cm[idx[r['true_name']], idx[r['pred_name']]] += 1
fig, ax = plt.subplots(figsize=(9,7))
im=ax.imshow(cm)
ax.set_xticks(range(len(classes))); ax.set_xticklabels(classes, rotation=60, ha='right')
ax.set_yticks(range(len(classes))); ax.set_yticklabels(classes)
ax.set_title('Confusion matrix')
for i in range(len(classes)):
    for j in range(len(classes)):
        if cm[i,j]: ax.text(j,i,str(cm[i,j]),ha='center',va='center',fontsize=7)
fig.colorbar(im, ax=ax, fraction=0.04)
fig.tight_layout(); fig.savefig(OUT_DIR/'confusion_matrix_recomputed.png', dpi=160); finish_analysis_figure()

fig, ax = plt.subplots(figsize=(9,4))
pred['pred_name'].value_counts().reindex(classes).plot(kind='bar', ax=ax)
ax.set_title('Prediction distribution')
ax.set_ylabel('count')
fig.tight_layout(); fig.savefig(OUT_DIR/'prediction_distribution_recomputed.png', dpi=160); finish_analysis_figure()



## Overlay viewer

This cell shows the saved wrong overlays. The filenames already encode the true and predicted classes.


In [ ]:

overlay_dir = BASE / 'figures' / 'wrong_overlays'
overlays = sorted(overlay_dir.glob('*.png'))
print('num overlays:', len(overlays))
for p in overlays[:12]:
    display(Markdown(f'### {p.name}'))
    display(Image(filename=str(p)))


## Findings from this diagnostic

After running the notebook, use the generated tables and plots above to summarize the v18 failure modes. In particular, compare v18 against v17 on:

1. **Fallback predictions**: whether low-margin predictions collapse into a simple class such as `boat`, especially when role overlap is weak.
2. **Edge-driven wrong decisions**: whether positive weighted-edge gaps still make incorrect classes win.
3. **Count and template bias**: whether count/template terms rescue or worsen difficult cases.
4. **Role evidence**: whether true-class role overlap is higher than predicted-class role overlap in remaining errors.
5. **New versus fixed errors**: whether v18 introduces new systematic confusions relative to v17 or fixes earlier v17 mistakes.

The comparison cell writes `new_errors_vs_previous.csv`, `fixed_errors_vs_previous.csv`, and `comparison_with_previous.json` into `OUT_DIR` when v17 diagnostics are available.



## Optional live-decode extension

If you run this notebook inside the repo with the grammar/checkpoint/cache available, add a live-decode cell that records, for both predicted and true parses:

```text
selected_pred_inst_edges
selected_true_inst_edges
selected_pred_missing_edges
selected_true_missing_edges
selected_pred_role_overlap
selected_true_role_overlap
selected_pred_support_overlap
selected_true_support_overlap
selected_pred_num_filled_slots
selected_true_num_filled_slots
```

These fields are not fully recoverable from the existing CSV zip alone. They should be added to future diagnostic notebooks and scripts.
